# ASG Airlines Data Engineering Pipeline

This notebook runs and documents the local end-to-end data pipeline for ASG Airlines.

It ingests the supplied flight, booking, payment, and passenger data; validates and cleans records; handles overnight flights; masks passenger identifiers; and creates curated datasets and KPI outputs for Power BI reporting.

## Pipeline objectives

The pipeline addresses the following data-quality and reporting requirements:

- Validate and standardize flight identifiers, airport codes, airlines, timestamps, and payment amounts.
- Remove exact duplicate flight records.
- Reject invalid flight records without silently modifying them.
- Correct overnight flights when arrival occurs before departure.
- Calculate flight duration, route, duration band, and anomaly indicators.
- Mask passenger identifiers and exclude direct PII from curated outputs.
- Produce reporting-ready dimensions, facts, KPI tables, quality metrics, and execution logs.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_FILE = PROJECT_ROOT / "data" / "raw" / "UseCase - Airlines.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Source file exists: {SOURCE_FILE.exists()}")
print(f"Output directory: {OUTPUT_DIR}")

## PII protection

The source data contains direct passenger identifiers, including names, email addresses, phone numbers, Aadhaar IDs, passport numbers, and emergency-contact information.

The pipeline uses a salted SHA-256 passenger key for curated outputs. Set `ASG_PII_SALT` outside source control before running the pipeline in a production environment.

In [ ]:
if not os.environ.get("ASG_PII_SALT"):
    print(
        "Warning: ASG_PII_SALT is not set. "
        "The local development fallback in pipeline.py will be used. "
        "Set a managed secret before production use."
    )
else:
    print("ASG_PII_SALT is configured.")

## Pipeline execution

The reusable implementation is stored in `src/pipeline.py`. Running it produces curated dimensions, facts, KPI tables, a quality summary, rejected-record output, and a pipeline log.

In [ ]:
pipeline_path = PROJECT_ROOT / "src" / "pipeline.py"

result = subprocess.run(
    [
        sys.executable,
        str(pipeline_path),
        "--input",
        str(SOURCE_FILE),
        "--output",
        str(OUTPUT_DIR),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pipeline execution failed.")

print("Pipeline execution completed successfully.")

## Data-quality results

The pipeline records source-row counts, duplicate removal, invalid-record rejection, overnight-flight corrections, anomaly counts, payment-quality issues, and unresolved booking-to-flight matches.

In [ ]:
quality_path = OUTPUT_DIR / "data_quality_summary.json"

with open(quality_path, encoding="utf-8") as file:
    quality = json.load(file)

pd.DataFrame(
    quality.items(),
    columns=["Quality measure", "Value"]
)

## Curated analytical model

The pipeline creates the following reporting-ready tables:

- `dim_flight`: flight schedule, route, corrected duration, duration band, and anomaly fields.
- `dim_passenger_masked`: masked passenger key, age band, gender, and date of birth.
- `fact_booking`: booking status, seat number, masked passenger key, and flight-match status.
- `fact_payment`: payment amount, payment method, and payment-status fields.
- KPI tables for route traffic, airline distribution, anomaly records, and summary metrics.

In [ ]:
flights = pd.read_csv(OUTPUT_DIR / "dim_flight.csv")
bookings = pd.read_csv(OUTPUT_DIR / "fact_booking.csv")
payments = pd.read_csv(OUTPUT_DIR / "fact_payment.csv")
passengers = pd.read_csv(OUTPUT_DIR / "dim_passenger_masked.csv")

summary = pd.DataFrame(
    [
        ["dim_flight", len(flights), len(flights.columns)],
        ["dim_passenger_masked", len(passengers), len(passengers.columns)],
        ["fact_booking", len(bookings), len(bookings.columns)],
        ["fact_payment", len(payments), len(payments.columns)],
    ],
    columns=["Table", "Rows", "Columns"],
)

summary

## Flight-duration and anomaly logic

Flight duration is calculated from corrected arrival time minus departure time.

If an arrival timestamp is earlier than its departure timestamp, the pipeline treats the record as an overnight flight and adds one day to the arrival timestamp. It then recalculates duration.

The pipeline flags:

- Short durations under 45 minutes.
- Long durations over 360 minutes.
- Arrival timestamps corrected for overnight flights.

The supplied source does not include scheduled-versus-actual timestamps. Therefore, anomaly results represent duration and data-quality exceptions, not on-time-performance delays.

In [ ]:
kpi_overview = pd.read_csv(OUTPUT_DIR / "kpi_overview.csv")
kpi_overview

In [ ]:
route_traffic = pd.read_csv(OUTPUT_DIR / "kpi_route_traffic.csv")
route_traffic.head(10)

In [ ]:
airline_distribution = pd.read_csv(
    OUTPUT_DIR / "kpi_airline_distribution.csv"
)

airline_distribution

In [ ]:
anomalies = pd.read_csv(OUTPUT_DIR / "kpi_anomalies.csv")

print(f"Flagged anomaly records: {len(anomalies)}")
anomalies.head(10)

## PII validation

The curated booking and passenger tables must not contain direct passenger identifiers. The validation below checks that restricted columns are absent.

In [ ]:
restricted_pii_columns = {
    "passenger_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "aadhaar_id",
    "passport_number",
    "emergency_contact_name",
    "emergency_contact_phone",
}

booking_pii = restricted_pii_columns.intersection(bookings.columns)
passenger_pii = restricted_pii_columns.intersection(passengers.columns)

assert not booking_pii, f"Restricted PII in fact_booking: {booking_pii}"
assert not passenger_pii, f"Restricted PII in dim_passenger_masked: {passenger_pii}"

print("PII validation passed: curated booking and passenger tables contain no direct PII.")

## Power BI handoff

Power BI imports the curated CSV files from `data/processed/`.

The primary report relationships are:

- `dim_passenger_masked[passenger_key]` to `fact_booking[passenger_key]`
- `dim_flight[flight_schedule_key]` to `fact_booking[flight_schedule_key]`
- `fact_booking[booking_id]` to `fact_payment[booking_id]`

The completed Power BI report contains Operations Summary, Duration Analysis, Route Performance, Airline Trends, and Delay and Anomaly Insights pages.

## Conclusion

The pipeline transforms raw ASG Airlines operational data into a validated, privacy-protected, reporting-ready dataset.

Outputs are stored in `data/processed/` and support the Power BI report, Excel dashboard, and documented case-study walkthrough.